In [ ]:
# Load packages
import xarray as xr 
import pandas as pd
import numpy  as np

import statsmodels.api as sm

import os


In [ ]:
################################
# Set the working directory
DATAPATH = '/Users/hao/EXTERNAL/PROGRESS/Carbon_Recovery_and_Climate'
WORKPATH = '/Users/hao/01-RESEARCH/PROGRESS/Carbon_Recovery_and_Climate'

cmip_models = [
    'ACCESS-ESM1-5',
    'CanESM5', 
    'CESM2',
    'CESM2-FV2',
    'CESM2-WACCM',
    'CESM2-WACCM-FV2',
    'CMCC-CM2-SR5',
    'CMCC-ESM2',
    #'E3SM-1-1',
    #'E3SM-1-1-ECA',
    'EC-Earth3-CC',
    'EC-Earth3-Veg-LR',
    'GFDL-ESM4',
    #'GISS-E2-1-G-CC',
    #'GISS-E2-2-H',
    'INM-CM4-8',
    'INM-CM5-0',
    'IPSL-CM5A2-INCA',
    'IPSL-CM6A-LR',
    'IPSL-CM6A-LR-INCA',
    'MPI-ESM-1-2-HAM',
    'MPI-ESM1-2-LR',
    'NorESM2-LM',
    'NorESM2-MM',
    'SAM0-UNICON',
    'TaiESM1'
]


In [ ]:
vars = ['nbp', 'mrso', 'nina34', 'fgco2', 'total_carbon_flux']

start = '1958-03-01'
end   = '2014-12-01'

for var in vars:
    print(f"*****\nProcessing variable: {var}")
    df = pd.read_csv(f'{WORKPATH}/01-data/{var}_CMIP.csv', index_col=0)
    df.index = pd.to_datetime(df.index)  # Ensure the index is datetime

    results = {}

    for model in cmip_models:
        print(f"{model}")

        # Step 1: Calculate mean seasonal cycle 
        mean_seasonal_cycle = df[model].groupby(df.index.month).mean().values

        full_seasonal_pattern = np.tile(mean_seasonal_cycle, len(df) // 12 + 1)[:len(df)]

        df_deseason_var = df[model].values - full_seasonal_pattern

        # Step 2: Calculate the annual growth rates by 12-month moving sum or mean
        if var in ['nbp', 'total_carbon_flux']:  # Variables that should be summed
            df_annual_var = pd.Series(df_deseason_var, index=df.index).rolling(window=12, center=False).sum()
        else:  # Variables like temperature or ENSO indices that should be averaged
            df_annual_var = pd.Series(df_deseason_var, index=df.index).rolling(window=12, center=False).mean()

        # Step 3: Remove the long-term trend using LOWESS
        trend = sm.nonparametric.lowess(endog=df_annual_var.values, exog=np.arange(len(df_annual_var)), frac=4/5, return_sorted=False)

        df_annual_detrended = df_annual_var.values - trend

        results[model] = df_annual_detrended  # Store result for each model

    # Convert results into a DataFrame for further analysis or saving
    df_result = pd.DataFrame(results, index=df.index)

    df_result = df_result.loc[start:end]  # Filter the DataFrame for the specified date range

    # Save the data
    df_result.to_csv(f'{WORKPATH}/01-data/{var}_CMIP_processed.csv')


*****
Processing variable: nbp
ACCESS-ESM1-5
CanESM5
CESM2
CESM2-FV2
CESM2-WACCM
CESM2-WACCM-FV2
CMCC-CM2-SR5
CMCC-ESM2
EC-Earth3-CC
EC-Earth3-Veg-LR
GFDL-ESM4
INM-CM4-8
INM-CM5-0
IPSL-CM5A2-INCA
IPSL-CM6A-LR
IPSL-CM6A-LR-INCA
MPI-ESM-1-2-HAM
MPI-ESM1-2-LR
NorESM2-LM
NorESM2-MM
SAM0-UNICON
TaiESM1
*****
Processing variable: mrso
ACCESS-ESM1-5
CanESM5
CESM2
CESM2-FV2
CESM2-WACCM
CESM2-WACCM-FV2
CMCC-CM2-SR5
CMCC-ESM2
EC-Earth3-CC
EC-Earth3-Veg-LR
GFDL-ESM4
INM-CM4-8
INM-CM5-0
IPSL-CM5A2-INCA
IPSL-CM6A-LR
IPSL-CM6A-LR-INCA
MPI-ESM-1-2-HAM
MPI-ESM1-2-LR
NorESM2-LM
NorESM2-MM
SAM0-UNICON
TaiESM1
*****
Processing variable: nina34
ACCESS-ESM1-5
CanESM5
CESM2
CESM2-FV2
CESM2-WACCM
CESM2-WACCM-FV2
CMCC-CM2-SR5
CMCC-ESM2
EC-Earth3-CC
EC-Earth3-Veg-LR
GFDL-ESM4
INM-CM4-8
INM-CM5-0
IPSL-CM5A2-INCA
IPSL-CM6A-LR
IPSL-CM6A-LR-INCA
MPI-ESM-1-2-HAM
MPI-ESM1-2-LR
NorESM2-LM
NorESM2-MM
SAM0-UNICON
TaiESM1
*****
Processing variable: fgco2
ACCESS-ESM1-5
CanESM5
CESM2
CESM2-FV2
CESM2-WACCM
CESM2-WACCM-FV2


/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


EC-Earth3-Veg-LR
GFDL-ESM4
INM-CM4-8
INM-CM5-0
IPSL-CM5A2-INCA
IPSL-CM6A-LR
IPSL-CM6A-LR-INCA
MPI-ESM-1-2-HAM
MPI-ESM1-2-LR
NorESM2-LM
NorESM2-MM
SAM0-UNICON
TaiESM1
*****
Processing variable: total_carbon_flux
ACCESS-ESM1-5
CanESM5


/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/hao/miniconda3/envs/climate/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


CESM2
CESM2-FV2
CESM2-WACCM
CESM2-WACCM-FV2
CMCC-CM2-SR5
CMCC-ESM2
EC-Earth3-CC
EC-Earth3-Veg-LR
GFDL-ESM4
INM-CM4-8
INM-CM5-0
IPSL-CM5A2-INCA
IPSL-CM6A-LR
IPSL-CM6A-LR-INCA
MPI-ESM-1-2-HAM
MPI-ESM1-2-LR
NorESM2-LM
NorESM2-MM
SAM0-UNICON
TaiESM1
